In [1]:
import json
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

import config
from src.db_io import leer_tabla_sqlite
from src.decisiones import decidir_perfil_incompleto, lift_condicional

df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")
(config.OUTPUTS_DIR / "eda").mkdir(parents=True, exist_ok=True)

universo = set(df["numero_id"])
sin_estimador = set(df.loc[df["falta_estimador"] == 1, "numero_id"])
# "nulos en las 5 columnas financieras": se reporta la lectura conservadora
# (algun nulo) y la estricta (los 5), porque las dos aparecen en el spec.
nulos_fin_any = set(df.loc[df[config.COLS_FINANCIERAS].isnull().any(axis=1), "numero_id"])
nulos_fin_all = set(df.loc[df[config.COLS_FINANCIERAS].isnull().all(axis=1), "numero_id"])
sin_vivienda = set(df.loc[df["tiene_dato_vivienda"] == 0, "numero_id"])

interseccion = sin_estimador & nulos_fin_any
# D7: el Jaccard es inaplicable (conjuntos desbalanceados, maximo alcanzable
# ~0.196 -- ver DECISIONES.md). Se usa lift condicional en su lugar.
lift_est_viv = lift_condicional(sin_estimador, sin_vivienda, universo)
decision_perfil = decidir_perfil_incompleto(lift_est_viv)

solapamiento = {
    "n_total": int(len(df)),
    "n_sin_estimador": len(sin_estimador),
    "n_nulos_financieros_any": len(nulos_fin_any),
    "n_nulos_financieros_all": len(nulos_fin_all),
    "n_interseccion_estimador_financieros": len(interseccion),
    "n_sin_vivienda": len(sin_vivienda),
    "lift_estimador_vivienda": lift_est_viv,
    "decision_perfil_incompleto": decision_perfil,
}
with open(config.OUTPUTS_DIR / "eda" / "faltantes_solapamiento.json", "w",
          encoding="utf-8") as f:
    json.dump(solapamiento, f, indent=2, ensure_ascii=False)

print(json.dumps(solapamiento, indent=2, ensure_ascii=False))
print(f"\n% de la base sin estimador: {len(sin_estimador)/len(df):.1%}")


{
  "n_total": 860223,
  "n_sin_estimador": 114431,
  "n_nulos_financieros_any": 260,
  "n_nulos_financieros_all": 249,
  "n_interseccion_estimador_financieros": 84,
  "n_sin_vivienda": 591691,
  "lift_estimador_vivienda": 1.0563759639070782,
  "decision_perfil_incompleto": {
    "crear_bandera_unica": false,
    "lift": 1.0563759639070782,
    "umbral": 1.5
  }
}

% de la base sin estimador: 13.3%


In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from src.decisiones import decidir_tratamiento_faltante_estimador
from src.features_modelo import features_modelo_a
from src.log_decisiones import registrar_decision

# Objetivo auxiliar: se puede predecir QUIEN no tiene estimador de ingreso?
y_falta = df["falta_estimador"]

# Predictoras: el resto de variables disponibles (financieras, de producto,
# demograficas). `features_modelo_a` ya excluye fuga (invesbot_/inv_virtual_,
# n_productos_total, etc.) referenciando directamente `fuga.COLUMNAS_FUGA_EXPLICITAS`
# como fuente unica de verdad, asi que se llama directo sobre `df.columns` sin
# prefiltro adicional en el notebook.
#
# Ademas de eso se quitan el propio estimador y sus banderas, que serian
# tautologicas para ESTE objetivo auxiliar en particular (no son fuga de la
# etiqueta de adopcion, `features_modelo_a` no las toca por eso):
#   - estimador_ingreso, tiene_estimador_ingreso: son literalmente el dato cuya
#     ausencia se quiere predecir.
#   - dif_ingreso_declarado_estimado (= ingresos_mensuales - estimador_ingreso,
#     ver src/derivadas.py) y pct_dif_ingreso (su version normalizada): se
#     calculan A PARTIR de estimador_ingreso, asi que son NaN exactamente
#     cuando falta_estimador==1 (verificado: 0 de los 114,431 clientes sin
#     estimador tienen dif_ingreso_declarado_estimado no nulo). Dejarlas entrar
#     no mide "hay patron real en variables independientes" -- deja que
#     HistGradientBoostingClassifier (que maneja NaN nativamente) separe las
#     clases con AUC=1.0 con un solo split en "es NaN esta columna", sin
#     aprender nada. Se comprobo empiricamente: incluirlas produce AUC=1.0000
#     con esa columna como #1 en importancia por un margen enorme. NO quitar
#     esta exclusion sin volver a verificar que ya no son tautologicas.
excluir = {"estimador_ingreso", "tiene_estimador_ingreso", "falta_estimador",
           "dif_ingreso_declarado_estimado", "pct_dif_ingreso"}
cols_aux = [c for c in features_modelo_a(df.columns) if c not in excluir]
X_aux = pd.get_dummies(
    df[cols_aux],
    columns=[c for c in ["desc_segmento", "grupo_edad", "desc_tipo_de_vivienda"]
             if c in cols_aux],
    dummy_na=False,
)

Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
    X_aux, y_falta, test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE, stratify=y_falta,
)
aux = HistGradientBoostingClassifier(random_state=config.RANDOM_STATE)
aux.fit(Xa_tr, ya_tr)
auc_falta = float(roc_auc_score(ya_te, aux.predict_proba(Xa_te)[:, 1]))

# Top-10 variables por permutation importance (mas confiable que la nativa).
# Se calcula sobre una submuestra del test para que el coste sea razonable.
sub = Xa_te.sample(n=min(30_000, len(Xa_te)), random_state=config.RANDOM_STATE)
pi = permutation_importance(
    aux, sub, ya_te.loc[sub.index], n_repeats=5,
    random_state=config.RANDOM_STATE, scoring="roc_auc",
)
top10 = (
    pd.DataFrame({"variable": sub.columns, "importancia": pi.importances_mean})
    .sort_values("importancia", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# HistGradientBoostingClassifier maneja NaN nativamente (Global Constraints)
decision = decidir_tratamiento_faltante_estimador(auc_falta, modelo_maneja_nulos=True)

print(f"AUC del clasificador auxiliar: {auc_falta:.4f}")
print(f"Conclusion: {decision['conclusion']}")
print(f"Accion a implementar: {decision['accion']}")
print("\nTop 10 variables asociadas a la ausencia del estimador:")
print(top10.to_string(index=False))

resultado_patron = {**decision, "top_10_variables": top10.to_dict(orient="records")}
with open(config.OUTPUTS_DIR / "eda" / "faltantes_deteccion_patron.json", "w",
          encoding="utf-8") as f:
    json.dump(resultado_patron, f, indent=2, ensure_ascii=False)

registrar_decision(
    clave="tratamiento_falta_estimador",
    decision=decision["accion"],
    motivo=f"AUC del clasificador auxiliar = {auc_falta:.4f} -> {decision['conclusion']} "
           f"(SPEC_V2 3.2). Modelo final maneja nulos nativamente.",
    evidencia={"auc": auc_falta,
               "top_10": top10["variable"].tolist(),
               "n_sin_estimador": len(sin_estimador)},
)


AUC del clasificador auxiliar: 0.8861
Conclusion: ausencia informativa
Accion a implementar: bandera_predictora_sin_imputacion_global

Top 10 variables asociadas a la ausencia del estimador:
                       variable  importancia
            saldo_liquido_total     0.090839
        bolsillos_saldo_prom_6m     0.055587
             ingresos_mensuales     0.045669
             bolsillos_tenencia     0.016301
                  total_activos     0.012622
         cuenta_ahorro_tenencia     0.012612
         bolsillos_tendencia_6m     0.009275
      antiguedad_relacion_meses     0.007933
        n_productos_no_etiqueta     0.006665
bolsillos_tendencia_relativa_6m     0.005718


WindowsPath('C:/Users/natam/OneDrive/Desktop/Prueba-Tecnica-CREAN/.claude/worktrees/pipeline-crean-sdd/outputs/decisiones/log_decisiones.csv')

In [3]:
# La accion NO se elige aqui: viene de la tabla de decision de 3.2 ya evaluada.
if decision["accion"] == "conservar_bandera_sin_imputar":
    print("ACCION: se conserva `falta_estimador` y `estimador_ingreso` queda NULO. "
          "El modelo final (HistGradientBoosting) maneja el nulo nativamente.")
    imputacion = None

elif decision["accion"] == "conservar_bandera_e_imputar_mediana_segmento":
    medianas = df.groupby("desc_segmento")["estimador_ingreso"].median()
    print("ACCION: imputacion por mediana de `desc_segmento`.")
    print(medianas.to_string())
    imputacion = {"tipo": "mediana_por_desc_segmento",
                  "valores": {str(k): float(v) for k, v in medianas.dropna().items()}}

else:  # "bandera_predictora_sin_imputacion_global"
    # 3.2, fila >0.70: la bandera es predictora de pleno derecho. NO se imputa
    # con medida central global. Se ofrece la mediana condicional al grupo que
    # las variables mas importantes identificaron.
    grupo = top10.loc[0, "variable"]
    print(f"ACCION: `falta_estimador` pasa a predictora de pleno derecho. "
          f"Sin imputacion global. Grupo condicional sugerido por la variable "
          f"mas asociada: {grupo}.")
    if "desc_segmento" in df.columns:
        medianas = df.groupby("desc_segmento")["estimador_ingreso"].median()
        print("Medianas condicionales disponibles por desc_segmento:")
        print(medianas.to_string())
    imputacion = {"tipo": "sin_imputacion_global", "grupo_sugerido": str(grupo)}

print(f"\nRegistro de imputacion: {imputacion}")


ACCION: `falta_estimador` pasa a predictora de pleno derecho. Sin imputacion global. Grupo condicional sugerido por la variable mas asociada: saldo_liquido_total.
Medianas condicionales disponibles por desc_segmento:
desc_segmento
personal        1.982166e+06
plus            6.124685e+06
preferencial    2.034356e+07

Registro de imputacion: {'tipo': 'sin_imputacion_global', 'grupo_sugerido': 'saldo_liquido_total'}


In [4]:
from scipy.stats import chi2_contingency

tabla = pd.crosstab(df["falta_estimador"], df["etiqueta_adopcion"])
chi2, p, gl, _ = chi2_contingency(tabla)

tasas = (
    df.groupby("falta_estimador")["etiqueta_adopcion"]
    .agg(n_clientes="count", n_adoptadores="sum", tasa_adopcion="mean")
    .reset_index()
)
tasas["grupo"] = tasas["falta_estimador"].map({0: "con estimador", 1: "sin estimador"})
dif = float(tasas.loc[tasas["falta_estimador"] == 1, "tasa_adopcion"].iloc[0]
            - tasas.loc[tasas["falta_estimador"] == 0, "tasa_adopcion"].iloc[0])

tasas["diferencia_vs_con_estimador"] = dif
tasas["chi2"] = float(chi2)
tasas["p_valor"] = float(p)
tasas["gl"] = int(gl)
tasas.to_csv(config.OUTPUTS_DIR / "eda" / "faltantes_tasa_adopcion.csv", index=False)

print(tasas.to_string(index=False))
print(f"\nDiferencia de tasa (sin - con): {dif:+.4%}")
print(f"Chi-cuadrado = {chi2:.2f}, gl = {gl}, p = {p:.3e}")
print(
    "\nSPEC_V2 3.4 -- RESTRICCION: no se elimina ningun cliente por falta de "
    f"estimador de ingresos. Son {len(sin_estimador):,} clientes "
    f"({len(sin_estimador)/len(df):.1%} de la base) y probablemente concentran el "
    "perfil de adquisicion en frio."
)


 falta_estimador  n_clientes  n_adoptadores  tasa_adopcion         grupo  diferencia_vs_con_estimador        chi2  p_valor  gl
               0      745792          61136       0.081975 con estimador                    -0.077605 8981.326719      0.0   1
               1      114431            500       0.004369 sin estimador                    -0.077605 8981.326719      0.0   1

Diferencia de tasa (sin - con): -7.7605%
Chi-cuadrado = 8981.33, gl = 1, p = 0.000e+00

SPEC_V2 3.4 -- RESTRICCION: no se elimina ningun cliente por falta de estimador de ingresos. Son 114,431 clientes (13.3% de la base) y probablemente concentran el perfil de adquisicion en frio.
